# Kaggle worker 1 — UColor
Runs UColor on UIEB and LSUI for selected seeds. The notebook preserves the common benchmark protocol and automatically completes the required ten-method/dataset smoke matrix unless a completed smoke baseline is restored.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/heniath/underwater-image-enhancement.git'
REPO_BRANCH = 'learnable-physics-extractor'
REPO_DIR = Path('/kaggle/working/underwater-image-enhancement')
UIEB_INPUT = Path('/kaggle/input/datasets/ohmahler91/uieb-dataset')
LSUI_INPUT = Path('/kaggle/input/datasets/ohmahler91/lsui-dataset/LSUI')
DATA_ROOT = Path('/kaggle/working/reference_data')
OUTPUT_ROOT = Path('/kaggle/working/reference_outputs_person1_ucolor')
TORCH_CACHE = Path('/kaggle/working/torch_cache')
SMOKE_BASELINE_INPUT = None  # e.g. Path('/kaggle/input/reference-smoke/reference_outputs')
METHODS = ['ucolor']
SEEDS = [0, 1, 2]  # Use one seed per account to parallelize further.
RUN_TESTS = True
RUN_WORKER = False
USE_RAM_CACHE = True

## Setup repository, environment, and attached datasets

In [ ]:
import csv, importlib, os, shutil, subprocess, sys

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only','origin',REPO_BRANCH], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[dev,profile,visualization]'], check=True)
os.chdir(REPO_DIR)
repo_src = str(REPO_DIR / 'src')
if repo_src not in sys.path: sys.path.insert(0, repo_src)
importlib.invalidate_caches()
import torch, uwir
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('GPU:', torch.cuda.get_device_name(0))
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('uwir:', Path(uwir.__file__).resolve())
TORCH_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['UWIR_TORCH_HOME'] = str(TORCH_CACHE)

def find_uieb(root):
    for candidate in [root] + [p.parent for p in root.rglob('raw-890')]:
        if (candidate/'raw-890').is_dir() and (candidate/'reference-890').is_dir(): return candidate.resolve()
    raise FileNotFoundError(f'UIEB layout not found below {root}')

def link(name, target):
    path = DATA_ROOT/name
    if path.is_symlink(): path.unlink()
    elif path.exists(): raise FileExistsError(f'Refusing to replace {path}')
    path.symlink_to(target, target_is_directory=True)

if not UIEB_INPUT.is_dir() or not LSUI_INPUT.is_dir(): raise FileNotFoundError('Attach both UIEB and LSUI, then correct their paths above.')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
link('UIEB', find_uieb(UIEB_INPUT)); link('LSUI', LSUI_INPUT.resolve())
if SMOKE_BASELINE_INPUT is not None and Path(SMOKE_BASELINE_INPUT).is_dir() and not OUTPUT_ROOT.exists():
    shutil.copytree(SMOKE_BASELINE_INPUT, OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('UIEB:', (DATA_ROOT/'UIEB').resolve()); print('LSUI:', (DATA_ROOT/'LSUI').resolve())

## Strict discovery and optional tests

In [ ]:
from uwir.datasets.lsui import discover_lsui
from uwir.datasets.uieb import discover_uieb
uieb = discover_uieb(DATA_ROOT/'UIEB')
lsui, report = discover_lsui(DATA_ROOT/'LSUI')
print('UIEB pairs:', len(uieb)); print('LSUI:', report)
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'], cwd=REPO_DIR, check=True)

## Smoke gate and assigned full runs
Set `RUN_WORKER=True`. To use several Kaggle accounts, give each account a disjoint `SEEDS` value such as `[0]`, `[1]`, or `[2]`. Never let accounts write to one shared output directory.

In [ ]:
from uwir.reference_methods import REFERENCE_METHODS

def command(mode, methods=None, seeds=None):
    cmd=[sys.executable,'-m','scripts.reference_methods_benchmark',f'--{mode}','--device','cuda','--data-root',str(DATA_ROOT),'--output-root',str(OUTPUT_ROOT)]
    if not USE_RAM_CACHE: cmd.append('--no-ram-cache')
    if methods: cmd += ['--methods', *methods]
    if seeds is not None: cmd += ['--seeds', *map(str,seeds)]
    print(' '.join(cmd)); subprocess.run(cmd, cwd=REPO_DIR, env=os.environ.copy(), check=True)

def smoke_complete():
    path=OUTPUT_ROOT/'smoke_results.csv'
    if not path.exists(): return False
    with path.open(newline='',encoding='utf-8') as handle:
        found={(r['dataset'],r['method']) for r in csv.DictReader(handle)}
    return found == {(d,m) for d in ('UIEB','LSUI') for m in REFERENCE_METHODS}

if RUN_WORKER:
    if not smoke_complete(): command('smoke')
    if not smoke_complete(): raise RuntimeError('The complete 10-combination smoke matrix did not pass.')
    command('full', METHODS, SEEDS)
else: print('Ready. Set RUN_WORKER=True to launch:', METHODS, SEEDS)

## Progress and handoff

In [ ]:
import pandas as pd
result=OUTPUT_ROOT/'per_run_results.csv'
if result.exists(): display(pd.read_csv(result).sort_values(['dataset','method','seed']))
print('Completed assigned runs:', len(list(OUTPUT_ROOT.glob('*/*/seed_*/test_metrics.json'))), '/', len(METHODS)*2*len(SEEDS))
print('Save as a private Kaggle Dataset:', OUTPUT_ROOT)